In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
from pathlib import Path
import json

In [2]:
#FILE LOADER
file_path= Path(r"C:\Users\revel\Downloads\New folder (2)\Data_sorted\ggH125_WW2lep.csv")
folder_path = file_path.parent
print(folder_path)


C:\Users\revel\Downloads\New folder (2)\Data_sorted


In [3]:
def inspect_csv(file_path):
    df = pd.read_csv(file_path)
    df["source_file"] = Path(file_path).name
    
    print(f"\nLoaded: {file_path}")
    print(f"Shape: {df.shape}")
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nDtypes:")
    print(df.dtypes)
    
    print("\nInfo:")
    df.info(memory_usage="deep")
    
    print("\nDescribe:")
    display(df.describe())
    
    return df

In [4]:
df = inspect_csv(file_path)
df["source_file"].head()


Loaded: C:\Users\revel\Downloads\New folder (2)\Data_sorted\ggH125_WW2lep.csv
Shape: (82695, 2)

Columns:
['mLL ptLL dPhi_LL dPhiLLmet MET mt goodjet_n goodbjet_n Lepton1_Pt Lepton1_Eta Lepton1_E Lepton1_Phi Lepton1_charge Lepton1_type Lepton2_Pt Lepton2_Eta Lepton2_E Lepton2_Phi Lepton2_charge Lepton2_type weight', 'source_file']

Dtypes:
mLL ptLL dPhi_LL dPhiLLmet MET mt goodjet_n goodbjet_n Lepton1_Pt Lepton1_Eta Lepton1_E Lepton1_Phi Lepton1_charge Lepton1_type Lepton2_Pt Lepton2_Eta Lepton2_E Lepton2_Phi Lepton2_charge Lepton2_type weight    object
source_file                                                                                                                                                                                                        object
dtype: object

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82695 entries, 0 to 82694
Data columns (total 2 columns):
 #   Column                                                                                 

,mLL ptLL dPhi_LL dPhiLLmet MET mt goodjet_n goodbjet_n Lepton1_Pt Lepton1_Eta Lepton1_E Lepton1_Phi Lepton1_charge Lepton1_type Lepton2_Pt Lepton2_Eta Lepton2_E Lepton2_Phi Lepton2_charge Lepton2_type weight,source_file
count,82695,82695
unique,82695,1
top,34.0927 52.2794 1.09927 2.48178 43.4527 90.183...,ggH125_WW2lep.csv
freq,1,82695


0    ggH125_WW2lep.csv
1    ggH125_WW2lep.csv
2    ggH125_WW2lep.csv
3    ggH125_WW2lep.csv
4    ggH125_WW2lep.csv
Name: source_file, dtype: object

In [5]:
def combine_csv_files(folder_path):
    csv_files = sorted(folder_path.glob("*.csv"))

    dfs = []

    for file in csv_files:
        print(f"Loading {file.name}")
        
        df = pd.read_csv(file)
        
        # add filename as label
        df["source_file"] = file.name
        
        dfs.append(df)

    # combine everything
    combined_df = pd.concat(dfs, ignore_index=True)

    print("\nDone.")
    print("Shape:", combined_df.shape)
    print("Files in dataset:", combined_df["source_file"].unique())
    return combined_df

In [6]:
combined_df = combine_csv_files(folder_path)

Loading ggH125_WW2lep.csv
Loading llll.csv
Loading lllv.csv
Loading llvv.csv
Loading lvvv.csv
Loading single_antitop_schan.csv
Loading single_antitop_tchan.csv
Loading single_antitop_wtchan.csv
Loading single_top_schan.csv
Loading single_top_tchan.csv
Loading single_top_wtchan.csv
Loading ttbar_lep.csv
Loading VBFH125_WW2lep.csv
Loading WlvZqq.csv
Loading Wminusenu.csv
Loading Wminusmunu.csv
Loading Wminustaunu.csv
Loading Wplusenu.csv
Loading Wplusmunu.csv
Loading Wplustaunu.csv
Loading WplvWmqq.csv
Loading WpqqWmlv.csv
Loading WqqZll.csv
Loading Zee.csv
Loading Zmumu.csv
Loading ZqqZll.csv
Loading Ztautau.csv

Done.
Shape: (258846, 2)
Files in dataset: ['ggH125_WW2lep.csv' 'llll.csv' 'lllv.csv' 'llvv.csv' 'lvvv.csv'
 'single_antitop_schan.csv' 'single_antitop_tchan.csv'
 'single_antitop_wtchan.csv' 'single_top_schan.csv' 'single_top_tchan.csv'
 'single_top_wtchan.csv' 'ttbar_lep.csv' 'VBFH125_WW2lep.csv' 'WlvZqq.csv'
 'Wminusenu.csv' 'Wminusmunu.csv' 'Wplusenu.csv' 'Wplusmunu.csv'
 '

In [7]:
combined_df.to_csv("combined_events_2.csv", index=False)

In [51]:

from sklearn.model_selection import train_test_split

def make_stratified_train_test_chunks(
    input_csv,
    train_size,
    test_size,
    label_col="source_file",
    seed=42,
    output_dir="stratified_split_output",
    save_files=True,
    overwrite=False
):
    """
    Load a CSV, perform a stratified shuffle split using `label_col`,
    and save train/test chunks plus metadata.

    Parameters
    ----------
    input_csv : str or Path
        Path to the combined CSV file.
    train_size : int
        Number of rows in the training chunk.
    test_size : int
        Number of rows in the testing chunk.
    label_col : str
        Column used for stratification.
    seed : int
        Random seed for reproducibility.
    output_dir : str or Path
        Directory for saving outputs.
    save_files : bool
        Whether to save train/test CSVs and metadata.
    overwrite : bool
        If False, raise an error when output files already exist.

    Returns
    -------
    train_df : pd.DataFrame
    test_df : pd.DataFrame
    """


    
    input_csv = Path(input_csv)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    train_csv = output_dir / "train_chunk.csv"
    test_csv = output_dir / "test_chunk.csv"
    meta_json = output_dir / "split_metadata.json"

    if save_files and not overwrite:
        for f in [train_csv, test_csv, meta_json]:
            if f.exists():
                raise FileExistsError(f"{f} already exists. Use overwrite=True to replace it.")

    df = pd.read_csv(input_csv)

    if label_col not in df.columns:
        raise ValueError(f"Column '{label_col}' not found in CSV.")

    n_total = len(df)
    n_needed = train_size + test_size

    if n_needed > n_total:
        raise ValueError(
            f"Requested {n_needed} rows total, but dataset only has {n_total} rows."
        )

    # First reduce to only the rows we want, while keeping stratification
    subset_df, _ = train_test_split(
        df,
        train_size=n_needed,
        stratify=df[label_col],
        random_state=seed,
        shuffle=True
    )

    # Then split that subset into train and test, again stratified
    train_df, test_df = train_test_split(
        subset_df,
        train_size=train_size,
        test_size=test_size,
        stratify=subset_df[label_col],
        random_state=seed,
        shuffle=True
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    if save_files:
        train_df.to_csv(train_csv, index=False)
        test_df.to_csv(test_csv, index=False)

        metadata = {
            "input_csv": str(input_csv),
            "label_column": label_col,
            "seed": seed,
            "total_rows_in_input": int(n_total),
            "rows_selected_total": int(n_needed),
            "train_size": int(len(train_df)),
            "test_size": int(len(test_df)),
            "train_output": str(train_csv),
            "test_output": str(test_csv)
        }

        with open(meta_json, "w") as f:
            json.dump(metadata, f, indent=4)

    print(f"Input rows: {n_total}")
    print(f"Selected rows: {n_needed}")
    print(f"Training rows: {len(train_df)}")
    print(f"Testing rows: {len(test_df)}")
    print(f"Seed: {seed}")

    print("\nTrain label distribution:")
    print(train_df[label_col].value_counts(normalize=True).sort_index())

    print("\nTest label distribution:")
    print(test_df[label_col].value_counts(normalize=True).sort_index())

    return train_df, test_df

In [55]:
train_df, test_df = make_stratified_train_test_chunks(
    input_csv="combined_events.csv",
    train_size=200000,
    test_size=20000,
    label_col="source_file",
    seed=12345,
    output_dir="atlas_stratified_split",
    save_files=True,
    overwrite=True
)

Input rows: 261309
Selected rows: 220000
Training rows: 200000
Testing rows: 20000
Seed: 12345

Train label distribution:
source_file
VBFH125_WW2lep.csv           0.131690
WlvZqq.csv                   0.000035
Wminusenu.csv                0.000065
Wminusmunu.csv               0.000065
Wplusenu.csv                 0.000070
Wplusmunu.csv                0.000110
WplvWmqq.csv                 0.000045
WpqqWmlv.csv                 0.000070
WqqZll.csv                   0.000035
Zee.csv                      0.000035
Zmumu.csv                    0.000160
ZqqZll.csv                   0.000030
Ztautau.csv                  0.000105
dataA.csv                    0.000490
dataB.csv                    0.002090
dataC.csv                    0.002865
dataD.csv                    0.003985
ggH125_WW2lep.csv            0.316465
llll.csv                     0.030835
lllv.csv                     0.080995
llvv.csv                     0.392865
lvvv.csv                     0.000020
single_antitop_schan.csv     0

In [62]:
import pandas as pd
import h5py
import numpy as np

def convert_train_chunk_csv_to_h5(csv_path, h5_path):
    """
    Convert a CSV of the form

    mLL ptLL ... weight,source_file
    28.6698 56.2186 ... 20.9643,ggH125_WW2lep.csv

    into a structured HDF5 file.

    The physics variables are space-separated, while source_file is separated
    by the final comma.
    """

    # Read the file using the comma as separator first:
    # - first column = all physics values as one string
    # - second column = source_file
    df_raw = pd.read_csv(csv_path, sep=",", dtype=str)

    # Split the first column on whitespace
    physics_col = df_raw.columns[0]
    physics_split = df_raw[physics_col].str.split(r"\s+", expand=True)

    # Column names are also stored inside that first header string
    physics_names = physics_col.split()

    if physics_split.shape[1] != len(physics_names):
        raise ValueError(
            f"Column count mismatch: found {physics_split.shape[1]} values per row, "
            f"but header defines {len(physics_names)} columns."
        )

    physics_split.columns = physics_names

    # Convert physics columns to numeric
    for col in physics_split.columns:
        physics_split[col] = pd.to_numeric(physics_split[col], errors="coerce")

    # Add source_file column
    physics_split["source_file"] = df_raw["source_file"]

    # Save to HDF5
    with h5py.File(h5_path, "w") as f:
        # Save numeric features as one 2D array
        numeric_cols = physics_names
        numeric_data = physics_split[numeric_cols].to_numpy(dtype=np.float32)
        f.create_dataset("features", data=numeric_data, compression="gzip")

        # Save column names
        f.create_dataset(
            "feature_names",
            data=np.array(numeric_cols, dtype="S")
        )

        # Save source_file as strings
        string_dt = h5py.string_dtype(encoding="utf-8")
        f.create_dataset(
            "source_file",
            data=physics_split["source_file"].to_numpy(dtype=object),
            dtype=string_dt,
            compression="gzip"
        )

    print(f"Saved HDF5 file to: {h5_path}")
    print(f"Shape of features: {numeric_data.shape}")
    print(f"Number of feature columns: {len(numeric_cols)}")


if __name__ == "__main__":
    convert_train_chunk_csv_to_h5("C:\\Users\\revel\\Downloads\\New folder (2)\\atlas_stratified_split\\test_chunk.csv", "C:\\Users\\revel\\Downloads\\New folder (2)\\test_chunk.h5")

Saved HDF5 file to: C:\Users\revel\Downloads\New folder (2)\test_chunk.h5
Shape of features: (20000, 21)
Number of feature columns: 21


In [ ]:
import h5py
import numpy as np
from pathlib import Path

# Labels to remove entirely
REMOVE_LABELS = np.array([13, 14, 15, 16])

# Input files
INPUT_FILES = [
    "train_binned_tokens_split_encoded.h5",
    "val_binned_tokens_split_encoded.h5",
    "test_binned_tokens_split_encoded.h5",
]

# Output suffix
OUTPUT_SUFFIX = "_filtered_binary"


def process_h5_file(input_path, remove_labels=REMOVE_LABELS, output_suffix=OUTPUT_SUFFIX):
    input_path = Path(input_path)

    if not input_path.exists():
        print(f"[SKIP] File not found: {input_path}")
        return

    output_path = input_path.with_name(f"{input_path.stem}{output_suffix}{input_path.suffix}")

    # ---- Load ----
    with h5py.File(input_path, "r") as f:
        if "features" not in f or "source_file" not in f:
            raise KeyError(f"{input_path} must contain datasets 'features' and 'source_file'")

        features = f["features"][:]
        source_file = f["source_file"][:]

    if len(features) != len(source_file):
        raise ValueError("Mismatch between features and labels length")

    # ---- Step 1: Remove unwanted labels ----
    mask = ~np.isin(source_file, remove_labels)
    features = features[mask]
    source_file = source_file[mask]

    # ---- Step 2: Convert to binary labels ----
    # 0 → 1 (signal)
    # everything else → 0 (background)
    binary_labels = (source_file == 0).astype(np.int8)

    # ---- Save ----
    with h5py.File(output_path, "w") as f:
        f.create_dataset("features", data=features, compression="gzip")
        f.create_dataset("source_file", data=binary_labels, compression="gzip")

    # ---- Logging ----
    print(f"\nProcessed: {input_path.name}")
    print(f"  Original rows : {len(mask)}")
    print(f"  Remaining rows: {len(features)}")
    print(f"  Signal (1)    : {np.sum(binary_labels == 1)}")
    print(f"  Background (0): {np.sum(binary_labels == 0)}")
    print(f"  Saved to      : {output_path.name}")


def main():
    for file in INPUT_FILES:
        process_h5_file(file)


if __name__ == "__main__":
    main()